In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pickle

from math import log, sqrt
from time import time
from pprint import pprint

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score as AUC, log_loss, accuracy_score as accuracy
from sklearn.metrics import (mean_squared_error as MSE, mean_absolute_error as MAE, r2_score as R2,
                             explained_variance_score as EVS)
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, MaxAbsScaler

from keras.models import Sequential
from keras.layers.core import Dense, Dropout
from keras.layers.normalization import BatchNormalization as BatchNorm
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.layers.advanced_activations import *
from keras.models import load_model

%load_ext autoreload
%autoreload 2

%matplotlib inline

plt.rcParams['figure.figsize'] = (10, 8)

/home/bulent/anaconda3/lib/python3.6/site-packages/h5py/__init__.py:34: FutureWarning: Conversion of the second argument of issubdtype from `float` to `np.floating` is deprecated. In future, it will be treated as `np.float64 == np.dtype(float).type`.
  from ._conv import register_converters as _register_converters
Using TensorFlow backend.


In [2]:
def fraction_within_eps(y_true, y_pred, epsilon=0.5):
    # fraction of entries where abs(y_true - y_pred) < epsilon
    
    count = np.sum(np.abs(y_true - y_pred) <= epsilon)
    return count / y_true.shape[0]

yt = np.array([10, 10, 10, 10, 10])
yp = np.array([9.2, 9.7, 10.3, 10.2, 11])
print(fraction_within_eps(yt, yp, 0.6))
FIE = fraction_within_eps

0.6


In [3]:
def concorr(x, y):
    # Return Lin's concordance correlation coefficient
    # x, y are numpy arrays
    
    xm = x.mean()
    ym = y.mean()
    xv = x.var()
    yv = y.var()
    xycov = np.sum((x-xm)*(y-ym)) / x.shape[0]
    lin = 2*xycov / (xv + yv + (xm - ym)**2)
    return lin

xx = np.random.randn(10)
yy = xx + 10
print('R2 corr coef is {} whereas concordance corr coef is {}'.format(R2(yy,xx), concorr(yy,xx)))

R2 corr coef is -70.16426422878456 whereas concordance corr coef is 0.027335749509432652


In [4]:
with open('stations-6to31.pkl', 'rb') as f:
    datas = pickle.load(f)
    
x_train6_ = datas['x_train6']
y_train6 = datas['y_train6']
x_train_ = datas['x_train']
y_train = datas['y_train']
x_dev_ = datas['x_dev']
y_dev = datas['y_dev']
x_test_ = datas['x_test']
y_test = datas['y_test']

print(f'x_train shape: {x_train_.shape}, y_train shape: {y_train.shape}')
print(f'x_train6 shape: {x_train6_.shape}, y_train6 shape: {y_train6.shape}')
print(f'x_dev shape: {x_dev_.shape}, y_dev shape: {y_dev.shape}')
print(f'x_test shape: {x_test_.shape}, y_test shape: {y_test.shape}')

x_train shape: (52416, 5), y_train shape: (52416,)
x_train6 shape: (10654, 5), y_train6 shape: (10654,)
x_dev shape: (9011, 5), y_dev shape: (9011,)
x_test shape: (16899, 5), y_test shape: (16899,)


From the best 21 configurations modes of respective categories are as follows.

**Initializer:** normal

**Layers:** 2

**Batch Size:** 64

**Optimizer:** adamax

**Shuffle:** True

**Scaler:** RobustScaler

**Loss:** mean_absolute_error

In [5]:
def scale_data(scaler, datas):
    # scaler is a scaling function from sklearn library
    # datas is a dictionary, containing 3 sets of x_data with keys - x_train, x_dev, x_test
    # fit on x_train and return the transformed sets of data
    
    x_train = scaler.fit_transform(datas['x_train'].astype(float))
    x_dev = scaler.transform(datas['x_dev'].astype(float))
    x_test = scaler.transform(datas['x_test'].astype(float))
    
    transformed = {'x_train': x_train, 'x_dev': x_dev, 'x_test': x_test}
    return transformed

data6_ = {'x_train': x_train6_, 'x_dev': x_dev_, 'x_test': x_test_}
data_ = {'x_train': x_train_, 'x_dev': x_dev_, 'x_test': x_test_}

data6 = scale_data(RobustScaler(), data6_)
data = scale_data(RobustScaler(), data_)

x_train6 = data6['x_train']
x_train = data['x_train']

x_dev6 = data6['x_dev']
x_dev = data['x_dev']

x_test6 = data6['x_test']
x_test = data['x_test']

print(f'x_train shape: {x_train.shape}, y_train shape: {y_train.shape}')
print(f'x_train6 shape: {x_train6.shape}, y_train6 shape: {y_train6.shape}')
print(f'x_dev shape: {x_dev.shape}, y_dev shape: {y_dev.shape}')
print(f'x_test shape: {x_test.shape}, y_test shape: {y_test.shape}')

x_train shape: (52416, 5), y_train shape: (52416,)
x_train6 shape: (10654, 5), y_train6 shape: (10654,)
x_dev shape: (9011, 5), y_dev shape: (9011,)
x_test shape: (16899, 5), y_test shape: (16899,)


In [6]:
def radstimator(h1=20, h2=15, num_vars=5):
    init = 'normal'
    
    model = Sequential()
    model.add( Dense( h1, kernel_initializer=init, input_dim=num_vars ))
    model.add( PReLU( alpha_initializer=init ))
    model.add( BatchNorm())
    model.add( Dense( h2, kernel_initializer=init ))
    model.add( PReLU( alpha_initializer=init ))
    model.add( Dropout( rate=0.4 ))
    
    model.add( Dense( 1, kernel_initializer=init, activation='linear' ))
    
    return model

In [9]:
print(x_train6_[:5]) # 'Latitude', 'BSH', 'Temperature(avg)', 'Daylength', 'H0'

[[40.141       5.9         4.22916667  9.19499685 13.69287119]
 [40.141       1.3         7.6375      9.20626964 13.74607822]
 [40.141       0.          6.2375      9.21860396 13.80432176]
 [40.141       0.          3.32916667  9.23198817 13.86758193]
 [40.141       0.7         5.06956522  9.24640976 13.93583675]]


In [7]:
data6_3v_ = {'x_train': x_train6_[:, [1, 3, 4]], 'x_dev': x_dev_[:, [1, 3, 4]], 'x_test': x_test_[:, [1, 3, 4]]}
data_3v_ = {'x_train': x_train_[:, [1, 3, 4]], 'x_dev': x_dev_[:, [1, 3, 4]], 'x_test': x_test_[:, [1, 3, 4]]}

data6_3v = scale_data(RobustScaler(), data6_3v_)
data_3v = scale_data(RobustScaler(), data_3v_)

x_train6_3v = data6_3v['x_train']
x_train_3v = data_3v['x_train']

x_dev6_3v = data6_3v['x_dev']
x_dev_3v = data_3v['x_dev']

x_test6_3v = data6_3v['x_test']
x_test_3v = data_3v['x_test']

print(f'x_train shape: {x_train_3v.shape}, y_train shape: {y_train.shape}')
print(f'x_train6 shape: {x_train6_3v.shape}, y_train6 shape: {y_train6.shape}')
print(f'x_dev shape: {x_dev_3v.shape}, y_dev shape: {y_dev.shape}')
print(f'x_test shape: {x_test_3v.shape}, y_test shape: {y_test.shape}')

x_train shape: (52416, 3), y_train shape: (52416,)
x_train6 shape: (10654, 3), y_train6 shape: (10654,)
x_dev shape: (9011, 3), y_dev shape: (9011,)
x_test shape: (16899, 3), y_test shape: (16899,)


In [10]:

validation_data6 = ( x_dev6_3v, y_dev )
for i in range(50):
    rads = radstimator(20, 15, 3)
    rads.compile(optimizer='adamax', loss='mean_absolute_error')

    early_stopping = EarlyStopping( monitor = 'val_loss', patience = 10, verbose = 0 )
    filepath = './6to31stats-2vars-h/6stations-h-nNH0 {}.h5'.format(i+1)
    checkpointer = ModelCheckpoint(filepath, monitor='val_loss', verbose=0, save_best_only=True )
    history = rads.fit( x_train6_3v, y_train6, epochs = 250, batch_size = 64, shuffle = True, 
                         validation_data = validation_data6, callbacks = [ early_stopping, checkpointer ], verbose=0)

    p = rads.predict( x_train6_3v, batch_size = 64 )

    mse = MSE( y_train6, p )
    rmse = sqrt( mse )
    mae = MAE( y_train6, p )
    r2 = R2( y_train6, p )
    evs = EVS( y_train6, p )
    fie_h = FIE( y_train6, np.squeeze(p), 0.5)
    fie_o = FIE( y_train6, np.squeeze(p), 1)
    fie_oh = FIE( y_train6, np.squeeze(p), 1.5)
    lin = concorr(y_train6, np.squeeze(p))
    
    print('C{:02d} » RMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}, '
              'FIE_H: {:.4f}, FIE_O: {:.4f}, FIE_OH: {:.4f}, LINCC: {:.4f}'.format(i+1,
                                                            rmse, mae, r2, evs, fie_h, fie_o, fie_oh, lin))

C01 » RMSE: 3.9805, MAE: 2.1804, R2: 0.8147, EVS: 0.8147, FIE_H: 0.2248, FIE_O: 0.4136, FIE_OH: 0.5538, LINCC: 0.8943
C02 » RMSE: 4.0645, MAE: 2.3638, R2: 0.8068, EVS: 0.8095, FIE_H: 0.1906, FIE_O: 0.3616, FIE_OH: 0.5007, LINCC: 0.8900
C03 » RMSE: 4.0448, MAE: 2.2891, R2: 0.8086, EVS: 0.8116, FIE_H: 0.1775, FIE_O: 0.3618, FIE_OH: 0.5131, LINCC: 0.8958
C04 » RMSE: 3.9845, MAE: 2.2678, R2: 0.8143, EVS: 0.8161, FIE_H: 0.1834, FIE_O: 0.3575, FIE_OH: 0.5099, LINCC: 0.8970
C05 » RMSE: 3.9751, MAE: 2.0761, R2: 0.8152, EVS: 0.8152, FIE_H: 0.2219, FIE_O: 0.4232, FIE_OH: 0.5903, LINCC: 0.9009
C06 » RMSE: 3.9707, MAE: 2.2623, R2: 0.8156, EVS: 0.8174, FIE_H: 0.1915, FIE_O: 0.3673, FIE_OH: 0.5208, LINCC: 0.8963
C07 » RMSE: 3.9783, MAE: 2.1779, R2: 0.8149, EVS: 0.8156, FIE_H: 0.1973, FIE_O: 0.3874, FIE_OH: 0.5465, LINCC: 0.8992
C08 » RMSE: 3.9847, MAE: 2.1837, R2: 0.8143, EVS: 0.8150, FIE_H: 0.2015, FIE_O: 0.3906, FIE_OH: 0.5504, LINCC: 0.8985
C09 » RMSE: 4.0279, MAE: 2.2266, R2: 0.8102, EVS: 0.8116

In [11]:
# Train with 6 stations data, and then with 31. Compare them.
# First for the variables n,N for H/H0.

validation_data = ( x_dev_3v, y_dev )
for i in range(50):
    rads = radstimator(20, 15, 3)
    rads.compile(optimizer='adamax', loss='mean_absolute_error')

    early_stopping = EarlyStopping( monitor = 'val_loss', patience = 10, verbose = 0 )
    filepath = './6to31stats-2vars-h/31stations-h-nNH0 {}.h5'.format(i+1)
    checkpointer = ModelCheckpoint(filepath, monitor='val_loss', verbose=0, save_best_only=True )
    history = rads.fit( x_train_3v, y_train, epochs = 250, batch_size = 64, shuffle = True, 
                         validation_data = validation_data6, callbacks = [ early_stopping, checkpointer ], verbose=0)

    p = rads.predict( x_train_3v, batch_size = 64 )

    mse = MSE( y_train, p )
    rmse = sqrt( mse )
    mae = MAE( y_train, p )
    r2 = R2( y_train, p )
    evs = EVS( y_train, p )
    fie_h = FIE( y_train, np.squeeze(p), 0.5)
    fie_o = FIE( y_train, np.squeeze(p), 1)
    fie_oh = FIE( y_train, np.squeeze(p), 1.5)
    lin = concorr(y_train, np.squeeze(p))
    
    print('C{:02d} » RMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}, '
              'FIE_H: {:.4f}, FIE_O: {:.4f}, FIE_OH: {:.4f}, LINCC: {:.4f}'.format(i+1,
                                                            rmse, mae, r2, evs, fie_h, fie_o, fie_oh, lin))

C01 » RMSE: 3.0889, MAE: 1.8038, R2: 0.8719, EVS: 0.8725, FIE_H: 0.2287, FIE_O: 0.4325, FIE_OH: 0.5977, LINCC: 0.9300
C02 » RMSE: 3.0734, MAE: 1.7437, R2: 0.8732, EVS: 0.8738, FIE_H: 0.2465, FIE_O: 0.4618, FIE_OH: 0.6271, LINCC: 0.9323
C03 » RMSE: 3.0724, MAE: 1.7735, R2: 0.8732, EVS: 0.8737, FIE_H: 0.2350, FIE_O: 0.4430, FIE_OH: 0.6098, LINCC: 0.9317
C04 » RMSE: 3.1155, MAE: 1.8491, R2: 0.8697, EVS: 0.8738, FIE_H: 0.2164, FIE_O: 0.4183, FIE_OH: 0.5862, LINCC: 0.9296
C05 » RMSE: 3.0714, MAE: 1.7641, R2: 0.8733, EVS: 0.8743, FIE_H: 0.2394, FIE_O: 0.4497, FIE_OH: 0.6143, LINCC: 0.9318
C06 » RMSE: 3.0820, MAE: 1.7516, R2: 0.8725, EVS: 0.8728, FIE_H: 0.2450, FIE_O: 0.4596, FIE_OH: 0.6219, LINCC: 0.9314
C07 » RMSE: 3.0762, MAE: 1.7703, R2: 0.8729, EVS: 0.8737, FIE_H: 0.2363, FIE_O: 0.4485, FIE_OH: 0.6132, LINCC: 0.9311
C08 » RMSE: 3.0803, MAE: 1.7909, R2: 0.8726, EVS: 0.8743, FIE_H: 0.2293, FIE_O: 0.4369, FIE_OH: 0.6063, LINCC: 0.9317
C09 » RMSE: 3.0934, MAE: 1.8137, R2: 0.8715, EVS: 0.8726